In [33]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [34]:
df = pd.read_csv("spam.csv", encoding="latin-1")

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [35]:
print("Dataset shape:", df.shape)

Dataset shape: (5572, 5)


In [36]:
df = df[["v1", "v2"]]

df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [37]:
df.columns = ["label", "text"]

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [38]:
df.isnull().sum()

,0
label,0
text,0


In [39]:
df["label"].value_counts()

,count
label,
ham,4825
spam,747


In [40]:
stemmer = PorterStemmer()

stop_words = set(ENGLISH_STOP_WORDS)


def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and numbers
    text = re.sub(r"[^a-z\s]", " ", text)

    # Tokenization
    tokens = text.split()

    # Remove stopwords
    tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    # Stemming
    tokens = [
        stemmer.stem(word)
        for word in tokens
    ]

    # Join tokens back into a sentence
    return " ".join(tokens)

In [41]:
df["clean_text"] = df["text"].apply(preprocess_text)

df.head()

,label,text,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...",jurong point crazi avail bugi great world la b...
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st t...
3,ham,U dun say so early hor... U c already then say...,dun say earli hor say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah don think goe usf live


In [42]:
df[["text", "clean_text"]].head(10)

,text,clean_text
0,"Go until jurong point, crazy.. Available only ...",jurong point crazi avail bugi great world la b...
1,Ok lar... Joking wif u oni...,ok lar joke wif oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st t...
3,U dun say so early hor... U c already then say...,dun say earli hor say
4,"Nah I don't think he goes to usf, he lives aro...",nah don think goe usf live
5,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey darl week word like fun tb ok xxx ...
6,Even my brother is not like to speak with me. ...,brother like speak treat like aid patent
7,As per your request 'Melle Melle (Oru Minnamin...,request mell mell oru minnaminungint nurungu v...
8,WINNER!! As a valued network customer you have...,winner valu network custom select receivea pri...
9,Had your mobile 11 months or more? U R entitle...,mobil month entitl updat latest colour mobil c...


In [43]:
X = df["clean_text"]
y = df["label"]

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [45]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4457
Testing samples: 1115


In [46]:
tfidf = TfidfVectorizer(
    max_features=5000
)

In [47]:
X_train_tfidf = tfidf.fit_transform(X_train)

In [48]:
X_test_tfidf = tfidf.transform(X_test)

In [49]:
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (4457, 5000)
Testing TF-IDF shape: (1115, 5000)


In [50]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [51]:
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [52]:
y_pred = model.predict(X_test_tfidf)

In [53]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9668161434977578


In [54]:
f1 = f1_score(
    y_test,
    y_pred,
    pos_label="spam"
)

print("F1-score:", f1)

F1-score: 0.8593155893536122


In [55]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       0.99      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115



In [56]:
feature_names = np.array(
    tfidf.get_feature_names_out()
)

coefficients = model.coef_[0]

In [57]:
top_spam_indices = np.argsort(
    coefficients
)[-10:][::-1]

top_spam_words = feature_names[
    top_spam_indices
]

print("Top 10 important words for SPAM:")

for word in top_spam_words:
    print(word)

Top 10 important words for SPAM:
txt
uk
claim
servic
mobil
www
repli
free
text
stop


In [58]:
top_ham_indices = np.argsort(
    coefficients
)[:10]

top_ham_words = feature_names[
    top_ham_indices
]

print("Top 10 important words for HAM:")

for word in top_ham_words:
    print(word)

Top 10 important words for HAM:
gt
lt
ok
ll
home
come
got
da
sorri
lor


In [59]:
top_words = pd.DataFrame({
    "HAM": top_ham_words,
    "SPAM": top_spam_words
})

top_words

,HAM,SPAM
0,gt,txt
1,lt,uk
2,ok,claim
3,ll,servic
4,home,mobil
5,come,www
6,got,repli
7,da,free
8,sorri,text
9,lor,stop


In [60]:
results = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Feature Extraction": ["TF-IDF"],
    "Accuracy": [accuracy],
    "F1 Score": [f1]
})

results

,Model,Feature Extraction,Accuracy,F1 Score
0,Logistic Regression,TF-IDF,0.966816,0.859316
